In [167]:
import numpy as np
import pandas as pd
from scipy.optimize import linprog

In [182]:
# we're going to have Python construct every relevant transmission constraint automatically, 
# across multiple operating scenarios.

buses = ['A','B', 'C', 'D']

lines = [
    ('A','B'),
    ('B','C'),
    ('C','D'),
    ('A','D'),
    ('B','D'),
]

K = np.zeros((len(lines), len(buses)) )

for i, (start, stop) in enumerate(lines):
    K[i, buses.index(start)] = 1
    K[i, buses.index(stop)] = -1


x = np.ones(5)

capacities = np.array([10, 9, 11, 10, 6])

costs = np.array([20, 65, 120])

d = np.array([0,0,5])

contingencies = [
    ("B", "D"),
    ("B", "C")
]

In [183]:


def calculate_ptdf(active_lines):
    
    K = np.zeros((len(active_lines), len(buses)) )

    for i, (start, stop) in enumerate(active_lines):
        K[i, buses.index(start)] = 1
        K[i, buses.index(stop)] = -1
    
    B = K.T @ K

    B_red = B[:(len(buses)-1), :(len(buses)-1)] # remove reference BUS row/ col

    thetas = np.linalg.solve(B_red, np.eye(len(buses)-1))
    thetas_full = np.vstack( (thetas, np.zeros(len(buses)-1)) )

    return( K @ thetas_full)

K2 = calculate_ptdf(active)

In [185]:
normal_ptdf = calculate_ptdf(lines)

bd_ptdf = calculate_ptdf(
    [line for line in lines if line != ("B", "D")]
)

bc_ptdf = calculate_ptdf(
    [line for line in lines if line != ("B", "C")]
)

capacities_bd = np.delete(capacities, 4)
capacities_bc = np.delete(capacities, 1)

A_list = []
b_list = []

ptdfs = [normal_ptdf, bd_ptdf, bc_ptdf]
caps = [capacities, capacities_bd, capacities_bc]

for ptdf, cap in zip(ptdfs, caps):
    # ptdf is a 2D matrix, -ptdf is a 2D matrix
    A_list.append(ptdf @ M)
    A_list.append(-ptdf @ M)
    
    # caps is a 1D array
    b_list.append(cap + ptdf @ d)
    b_list.append(cap + ptdf @ d)


A_ub = np.vstack(A_list) 
b_ub = np.concatenate(b_list)



NameError: name 'M' is not defined

In [174]:
base_lines = ["AB", "BC", "CD", "AD", "BD"]

lines_normal = base_lines[:] # Copy all 5
lines_bd = base_lines[:]
del lines_bd[4] 

lines_bc = base_lines[:]
del lines_bc[1] 

scenario = ["Normal", "B-D Outage", "B-C Outage"]
line_list = [lines_normal, lines_bd, lines_bc]

constraint_data = []

for scenarios, active_line in zip(scenario, line_list):
    for line in active_line:
        constraint_data.append({
            "Scenario": scenarios,
            "Monitored Line": line,
            "Direction": "Positive"
        })
    for line in active_line:
        constraint_data.append({
            "Scenario": scenarios,
            "Monitored Line": line,
            "Direction": "Negative"
        })


#zip(scenario_prefixes, scenario_line_lists)

constraints = pd.DataFrame(constraint_data)

In [177]:


# Energy balance: g_A + b_B + g_C = 16
A_eq = [[1, 1, 1]]
b_eq = [16]
bounds = [(0, 16), (0, 16), (0,16)]

result = linprog(
    c=costs,
    A_ub=A_ub,
    b_ub=b_ub,
    A_eq=A_eq,
    b_eq=b_eq,
    bounds=bounds,
    method="highs"
)


constraints["Shadow Price"] = -result.ineqlin.marginals
constraints["Slack"] = result.ineqlin.residual

binding = constraints.loc[np.isclose(constraints['Slack'],0),]
print(binding)



      Scenario Monitored Line Direction  Shadow Price  Slack
13  B-D Outage             AD  Positive    193.333333    0.0
21  B-C Outage             BD  Positive     10.000000    0.0


In [ ]:

# print(np.where(np.isclose(result.ineqlin.residual, 0)))


# print(result.ineqlin.marginals[[13, 21]])
# print(result.eqlin.marginals)


# print("Optimal dispatch:", result.x)
# print("Total cost:", result.fun)

# print("Binding constraint indices:")
# print(np.where(np.isclose(result.ineqlin.residual, 0))[0])

# print("Nonzero transmission marginals:")
# print(np.where(~np.isclose(result.ineqlin.marginals, 0))[0])

LMP_D = 168.33333333
opt_gen = np.array([10,4,2])
# BD outage
LMP_B = LMP_D + A_ub[13,1] * -193.33333333 + A_ub[21,1] * -10
print(f"B: { LMP_B}")

LMP_A = LMP_D + A_ub[13,0] * -193.33333333 + A_ub[21,0] * -10
print(f"A: {LMP_A}")

LMP_C = LMP_D + A_ub[13,2] * -193.33333333 + A_ub[21,2] * -10
print(f"C: {LMP_C}")


[-193.33333333  -10.        ]
[168.33333333]
B: 64.99999999833332
A: 19.999999999166658
C: 119.9999999975
